# Inter-Annotator-Agreement: Pronomen-Ebene

**Ordnerkontext:** liegt in `00_werkzeuge/`, liest aus
`03_annotation_laufend/pronomen/`, schreibt nach `05_ergebnisse/iaa/`.

## Konfiguration


In [ ]:
from pathlib import Path

BASIS_ARBEIT = Path("..") / "03_annotation_laufend" / "pronomen"
AUSGABE      = Path("..") / "05_ergebnisse" / "iaa"

HAUPTERHEBUNG = "A4"                                # Person der Haupterhebung
CSV_HAUPT = BASIS_ARBEIT / f"{HAUPTERHEBUNG}_pronomen_final.csv"   # = Gold

SONDERWERTE = {"expletiv", "keine_figur", "unklar", "kein_pronomen"}
GROSSKLEIN_EGAL = True        # Hauptauswertung ohne Schreibweisenunterschiede
BOOTSTRAP_REPS, SEED = 2000, 42

#Ab welcher Differenz gilt die Modellnähe einer Person als auffällig?
MODELL_SCHWELLE = 0.25

#Anlass dieses Durchlaufs, erscheint in Dateien und Verlaufstabelle
LAUF_ETIKETT = "vor Adjudikation"

AUSGABE.mkdir(parents=True, exist_ok=True)
assert CSV_HAUPT.exists(), f"Fehlt: {CSV_HAUPT}"

zweit = sorted(BASIS_ARBEIT.glob("*_iaa_final.csv"))
assert zweit, "Keine Zweitbearbeitung (*_iaa_final.csv) gefunden."

dateien = {HAUPTERHEBUNG: CSV_HAUPT}
for p in zweit:
    dateien[p.name.split("_")[0]] = p

LAUF_NR = len(sorted(AUSGABE.glob("iaa_pronomen_lauf*.json"))) + 1
print(f"Menschliche Bearbeitende ({len(dateien)}):")
for name, p in dateien.items():
    print(f"  {name:<5} {p}")
modell_dateien = {p.stem.replace("modell_", ""): p
                  for p in sorted(BASIS_ARBEIT.glob("modell_*.csv"))}
if modell_dateien:
    print(f"Modell-Referenzen ({len(modell_dateien)}): "
          + ", ".join(modell_dateien))
else:
    print("Modell-Referenzen: keine (Schritt 5 entfaellt)")
print(f"Dieser Durchlauf: Lauf {LAUF_NR} ({LAUF_ETIKETT})")

Menschliche Bearbeitende (3):
  A4    ../03_annotation_laufend/pronomen/A4_pronomen_final.csv
  A2    ../03_annotation_laufend/pronomen/A2_pronomen_iaa_final.csv
  A5    ../03_annotation_laufend/pronomen/A5_pronomen_iaa_final.csv
Modell-Referenzen (3): chatgpt-gpt5.6-luna_final, claude-fable5_final, claude-opus5_final
Dieser Durchlauf: Lauf 1 (vor Adjudikation)


## Schritt 1 — Maße implementieren und selbst validieren


In [ ]:
from collections import Counter
from itertools import permutations

def prozent_agreement(einheiten):
    """Anteil uebereinstimmender Paare, gemittelt ueber die Einheiten."""
    werte = []
    for urteile in einheiten:
        u = [x for x in urteile if x is not None]
        paare = list(permutations(range(len(u)), 2))
        werte.append(sum(1 for i, j in paare if u[i] == u[j]) / len(paare))
    return sum(werte) / len(werte)


def einheit_agreement(urteile):
    """Uebereinstimmungsanteil INNERHALB einer Einheit (0 bis 1)."""
    u = [x for x in urteile if x is not None]
    paare = list(permutations(range(len(u)), 2))
    return sum(1 for i, j in paare if u[i] == u[j]) / len(paare)


def fleiss_kappa(einheiten):
    umfaenge = {len([x for x in u if x is not None]) for u in einheiten}
    if len(umfaenge) > 1:
        raise ValueError(
            "Fleiss' kappa verlangt gleich viele Urteile je Einheit; "
            f"vorgefunden: {sorted(umfaenge)}. Messgrundlage pruefen.")
    p_o = prozent_agreement(einheiten)
    zaehler, gesamt = Counter(), 0
    for urteile in einheiten:
        for x in (v for v in urteile if v is not None):
            zaehler[x] += 1
            gesamt += 1
    p_e = sum((v / gesamt) ** 2 for v in zaehler.values())
    return ((p_o - p_e) / (1 - p_e) if p_e < 1 else float("nan")), p_o, p_e


def kategorienverteilung(einheiten):
    """Anteil jeder Kategorie an allen abgegebenen Urteilen."""
    zaehler, gesamt = Counter(), 0
    for urteile in einheiten:
        for x in (v for v in urteile if v is not None):
            zaehler[x] += 1
            gesamt += 1
    return {k: v / gesamt for k, v in zaehler.most_common()}


def lesehilfe_kappa(einheiten, kappa, beobachtet, erwartet, einrueckung=""):
    """Gibt die Rechenschritte hinter Fleiss' kappa aus."""
    vert = kategorienverteilung(einheiten)
    gezeigt = list(vert.items())[:6]
    rest = len(vert) - len(gezeigt)
    print(f"{einrueckung}Kategorien ({len(vert)}): "
          + " | ".join(f"{k} {100 * v:.1f}%" for k, v in gezeigt)
          + (f" | ... und {rest} weitere" if rest > 0 else ""))
    print(f"{einrueckung}erwartet = Summe der quadrierten Anteile = "
          f"{erwartet:.4f}")
    print(f"{einrueckung}kappa = ({beobachtet:.4f} - {erwartet:.4f}) / "
          f"(1 - {erwartet:.4f}) = {beobachtet - erwartet:.4f} / "
          f"{1 - erwartet:.4f} = {kappa:.4f}")


def fleiss_gegenprobe(einheiten):
    try:
        import numpy as np
        from statsmodels.stats.inter_rater import (aggregate_raters,
                                                   fleiss_kappa as sm_fleiss)
    except ImportError:
        return None
    tabelle, _ = aggregate_raters(np.array(einheiten))
    return float(sm_fleiss(tabelle, method="fleiss"))


# Anker 1: Fleiss, 30 Einheiten und 6 Bearbeitende, wie im
# Testfall der Bibliothek DKPro Agreement
referenz = [(4, 4, 4, 4, 4, 4), (2, 2, 2, 5, 5, 5), (2, 3, 3, 3, 3, 5),
            (5, 5, 5, 5, 5, 5), (2, 2, 2, 4, 4, 4), (1, 1, 3, 3, 3, 3),
            (3, 3, 3, 3, 5, 5), (1, 1, 3, 3, 3, 4), (1, 1, 4, 4, 4, 4),
            (5, 5, 5, 5, 5, 5), (1, 4, 4, 4, 4, 4), (1, 2, 4, 4, 4, 4),
            (2, 2, 2, 3, 3, 3), (1, 4, 4, 4, 4, 4), (2, 2, 4, 4, 4, 5),
            (3, 3, 3, 3, 3, 5), (1, 1, 1, 4, 5, 5), (1, 1, 1, 1, 1, 2),
            (2, 2, 4, 4, 4, 4), (1, 3, 3, 5, 5, 5), (5, 5, 5, 5, 5, 5),
            (2, 4, 4, 4, 4, 4), (2, 2, 4, 5, 5, 5), (1, 1, 4, 4, 4, 4),
            (1, 4, 4, 4, 4, 5), (2, 2, 2, 2, 2, 4), (1, 1, 1, 1, 5, 5),
            (2, 2, 4, 4, 4, 4), (1, 3, 3, 3, 3, 3), (5, 5, 5, 5, 5, 5)]
fk, po, pe = fleiss_kappa(referenz)
print("Selbstvalidierung — Referenzbeispiel mit 6 Bearbeitenden")
print("  (Soll: 0,5556 / 0,2201 / 0,430)")
print(f"  beobachtet {po:.4f} | erwartet {pe:.4f} | Fleiss kappa {fk:.4f}")
for wert, soll, name in ((po, 0.5556, "beobachtet"),
                         (pe, 0.2201, "erwartet"), (fk, 0.430, "kappa")):
    assert abs(wert - soll) < 0.001, f"{name}: {wert:.4f} statt {soll}"

fremd = fleiss_gegenprobe(referenz)
if fremd is None:
    print("  Gegenprobe uebersprungen: statsmodels nicht installiert")
else:
    assert abs(fk - fremd) < 1e-9, (fk, fremd)
    print(f"  Gegenprobe statsmodels: {fremd:.4f} — uebereinstimmend")

try:
    fleiss_kappa([(1, 1, 1), (1, 1)])
except ValueError:
    print("  Ungleiche Urteilszahlen werden erkannt")
else:
    raise AssertionError("ungleiche Urteilszahlen blieben unbemerkt")

# Anker 2 und 3: Grenzfaelle, deren Ergebnis feststeht
#   identische Reihen bei zwei Kategorien -> kappa = 1
#   Uebereinstimmung genau auf Zufallsniveau -> kappa = 0
assert abs(fleiss_kappa([("a", "a"), ("b", "b"),
                         ("a", "a"), ("b", "b")])[0] - 1.0) < 1e-9
assert abs(fleiss_kappa([("x", "x"), ("x", "y"),
                         ("y", "x"), ("y", "y")])[0]) < 1e-9
print("  Fleiss kappa: Grenzfaelle 1,0 und 0,0 reproduziert")

# Anker 4: Uebereinstimmung innerhalb einer einzelnen Einheit. Diese
# Funktion bestimmt spaeter, welche Einheiten als einstimmig gelten und
# damit, was in die Divergenz- und Kommentarlisten gelangt.
assert einheit_agreement(("a", "a", "a")) == 1.0
assert abs(einheit_agreement(("a", "a", "b")) - 1 / 3) < 1e-9
assert einheit_agreement(("a", "b", "c")) == 0.0
print("  einheit_agreement: 1,000 / 0,333 / 0,000 korrekt")

# Anker: Kategorienverteilung, aus der die erwartete Uebereinstimmung folgt
_v = kategorienverteilung([("a", "a"), ("a", "b")])
assert abs(_v["a"] - 0.75) < 1e-9 and abs(_v["b"] - 0.25) < 1e-9
assert abs(sum(x * x for x in _v.values()) - 0.625) < 1e-9
print("  kategorienverteilung: 0,75 / 0,25 korrekt")
print()
print("Alle Anker getroffen — die Implementierung darf verwendet werden.")

Selbstvalidierung — Referenzbeispiel mit 6 Bearbeitenden
  (Soll: 0,5556 / 0,2201 / 0,430)
  beobachtet 0.5556 | erwartet 0.2199 | Fleiss kappa 0.4302
  Gegenprobe statsmodels: 0.4302 — uebereinstimmend
  Ungleiche Urteilszahlen werden erkannt
  Fleiss kappa: Grenzfaelle 1,0 und 0,0 reproduziert
  einheit_agreement: 1,000 / 0,333 / 0,000 korrekt
  kategorienverteilung: 0,75 / 0,25 korrekt

Alle Anker getroffen — die Implementierung darf verwendet werden.


## Schritt 2 — Daten laden und Messgrundlage bestimmen


In [3]:
import pandas as pd

tabellen = {}
for name, pfad in dateien.items():
    tabellen[name] = pd.read_csv(pfad, dtype=str, keep_default_na=False)
    print(f"  {name}: {len(tabellen[name])} Zeilen")

schluessel = ["erzaehlung", "satzindex", "token_nr"]
basis = tabellen[HAUPTERHEBUNG]
for name, tab in tabellen.items():
    assert tab[schluessel].equals(basis[schluessel]), \
        f"{name} ist nicht zeilenidentisch zu A3 — gleiche Vorlage verwendet?"

personen = list(dateien.keys())
paar = basis[["sammlung", "erzaehlung", "tranche", "block", "satzindex",
              "token_nr", "pronomen", "typ", "satz_markiert"]].copy()
for name in personen:
    paar[name] = tabellen[name]["referent_gold"].str.strip()

zweitpersonen = [p for p in personen if p != HAUPTERHEBUNG]
bearbeitet = paar[(paar[zweitpersonen] != "").any(axis=1)]
vollstaendig = paar[(paar[personen] != "").all(axis=1)].copy()

print()
print(f"Von mindestens einer Zweitperson bearbeitet: {len(bearbeitet)}")
print(f"Von allen {len(personen)} Beteiligten bearbeitet: "
      f"{len(vollstaendig)}  <- Messgrundlage")
for name in personen:
    fehlt = int((bearbeitet[name] == "").sum())
    if fehlt:
        print(f"  Luecken bei {name}: {fehlt} (aus der Messung ausgeschlossen)")

erz = sorted(vollstaendig["erzaehlung"].unique())
print()
print(f"Erzaehlungen der Messgrundlage ({len(erz)}):")
for e in erz:
    teil = vollstaendig[vollstaendig["erzaehlung"] == e]
    print(f"  [{teil['sammlung'].iloc[0]:<10}] {e:<55} {len(teil):>3}")

  A4: 248 Zeilen
  A2: 248 Zeilen
  A5: 248 Zeilen

Von mindestens einer Zweitperson bearbeitet: 49
Von allen 3 Beteiligten bearbeitet: 49  <- Messgrundlage

Erzaehlungen der Messgrundlage (4):
  [Arabisch  ] 0_Die Geschichte von Ali dem Kairenser und dem spukenden Hause in Bagdad  12
  [Grimm     ] 1_Die drei Spinnerinnen                                  14
  [Andersen  ] 2_Des Kaisers neue Kleider                               11
  [Japanisch ] 3_Das Mädchen mit dem Holznapfe                          12


## Schritt 3 — Gesamtübereinstimmung


In [4]:
def norm(v):
    return " ".join(v.split()).casefold()

for name in personen:
    vollstaendig["n_" + name] = vollstaendig[name].map(norm)

def spaltennamen(normalisiert):
    return [("n_" + p) if normalisiert else p for p in personen]

def kennzahlen(normalisiert):
    sn_ = spaltennamen(normalisiert)
    einheiten = [tuple(r) for r in vollstaendig[sn_].to_numpy()]
    fk_wert, po_wert, pe_wert = fleiss_kappa(einheiten)
    einstimmig = sum(1 for u in einheiten if len(set(u)) == 1)
    return dict(n=len(einheiten), prozent=po_wert, fleiss_kappa=fk_wert,
                beobachtet=po_wert, erwartet=pe_wert,
                einstimmig=einstimmig,
                uneinig=len(einheiten) - einstimmig)

streng = kennzahlen(False)
normal = kennzahlen(True)

print(f"{'Lesart':<16}{'n':>5}{'PA':>9}{'Fleiss K':>10}"
      f"{'einstimmig':>12}{'uneinig':>9}")
for name, k in (("streng", streng), ("normalisiert", normal)):
    print(f"{name:<16}{k['n']:>5}{k['prozent']:>9.3f}"
          f"{k['fleiss_kappa']:>10.3f}{k['einstimmig']:>12}"
          f"{k['uneinig']:>9}")
print(f"{'erwartet (Zufall)':<16}{'':>5}{normal['erwartet']:>9.3f}")

nur_schreibweise = streng["uneinig"] - normal["uneinig"]
print()
print(f"Davon reine Schreibweisenunterschiede: {nur_schreibweise}")
haupt = normal if GROSSKLEIN_EGAL else streng
print(f"Hauptauswertung: {'normalisiert' if GROSSKLEIN_EGAL else 'streng'} "
      f"(Fleiss kappa = {haupt['fleiss_kappa']:.3f})")

einheiten_haupt = [tuple(r) for r in
                   vollstaendig[spaltennamen(GROSSKLEIN_EGAL)].to_numpy()]
print()
lesehilfe_kappa(einheiten_haupt, haupt["fleiss_kappa"], haupt["beobachtet"],
                haupt["erwartet"], einrueckung="  ")

gegen = fleiss_gegenprobe(einheiten_haupt)
if gegen is None:
    print("Gegenprobe uebersprungen: statsmodels nicht installiert")
else:
    assert abs(haupt["fleiss_kappa"] - gegen) < 1e-9
    print(f"Gegenprobe statsmodels: {gegen:.3f} — uebereinstimmend")

Lesart              n       PA  Fleiss K  einstimmig  uneinig
streng             49    0.422     0.379           8       41
normalisiert       49    0.871     0.858          40        9
erwartet (Zufall)         0.090

Davon reine Schreibweisenunterschiede: 32
Hauptauswertung: normalisiert (Fleiss kappa = 0.858)

  Kategorien (24): keine_figur 19.7% | ali 10.2% | weiber 10.2% | mädchen 8.2% | sohn 8.2% | minister 6.1% | ... und 18 weitere
  erwartet = Summe der quadrierten Anteile = 0.0903
  kappa = (0.8707 - 0.0903) / (1 - 0.0903) = 0.7805 / 0.9097 = 0.8579
Gegenprobe statsmodels: 0.858 — uebereinstimmend


## Schritt 4 — Paarweise Werte und Abweichungsindex



In [ ]:
from itertools import combinations

sn = spaltennamen(GROSSKLEIN_EGAL)
spalte = dict(zip(personen, sn))

paarwerte = {}

print(f"{'Paar':<14}{'PA':>9}{'F-K = pi':>10}{'uneinig':>9}")
for a, b in combinations(personen, 2):
    va = list(vollstaendig[spalte[a]])
    vb = list(vollstaendig[spalte[b]])
    pa_ab = sum(1 for x, y in zip(va, vb) if x == y) / len(va)
    fk_ab, _, _ = fleiss_kappa(list(zip(va, vb)))
    un = sum(1 for x, y in zip(va, vb) if x != y)
    paarwerte[f"{a}-{b}"] = dict(prozent=pa_ab, fleiss_kappa=fk_ab,
                                 uneinig=un)
    print(f"{a + '-' + b:<14}{pa_ab:>9.3f}{fk_ab:>10.3f}{un:>9}")

index = {}
if len(personen) > 2:
    print()
    print("Abweichungsindex (mittlere Uebereinstimmung mit den anderen):")
    for p in personen:
        werte = [v["prozent"] for k, v in paarwerte.items()
                 if p in k.split("-")]
        index[p] = sum(werte) / len(werte)
    for p, v in sorted(index.items(), key=lambda x: x[1]):
        print(f"  {p:<5} {v:.3f}")
    schlechteste = min(index, key=index.get)
    spanne = max(index.values()) - min(index.values())
    if spanne > 0.05:
        print(f"  Hinweis: {schlechteste} weicht deutlich ab (Spanne "
              f"{spanne:.3f}) — Richtlinienverstaendnis gezielt pruefen.")
    else:
        print(f"  Kein Ausreisser erkennbar (Spanne {spanne:.3f}).")
else:
    print()
    print("Abweichungsindex entfaellt: nur zwei Bearbeitende.")

Paar                 PA  F-K = pi  uneinig
A4-A2             0.918     0.911        4
A4-A5             0.837     0.820        8
A2-A5             0.857     0.842        7

Abweichungsindex (mittlere Uebereinstimmung mit den anderen):
  A5    0.847
  A4    0.878
  A2    0.888
  Kein Ausreisser erkennbar (Spanne 0.041).


## Schritt 5 — Modellnähe-Diagnose

In [ ]:
modellnaehe = {}
if not modell_dateien:
    print("Keine Modelldatei vorhanden — Diagnose uebersprungen.")
else:
    modell_spalten = {}
    for mname, mpfad in modell_dateien.items():
        md = pd.read_csv(mpfad, dtype=str, keep_default_na=False)
        assert md[schluessel].equals(basis[schluessel]), \
            f"Modelldatei {mpfad.name} ist nicht zeilenidentisch zur Vorlage."
        roh = "M_" + mname
        vollstaendig[roh] = md["referent_gold"].str.strip().loc[vollstaendig.index]
        if GROSSKLEIN_EGAL:
            vollstaendig["n_" + roh] = vollstaendig[roh].map(norm)
        modell_spalten[mname] = ("n_" + roh) if GROSSKLEIN_EGAL else roh

    schwer_maske = [len({r[s] for s in sn}) > 1
                    for _, r in vollstaendig.iterrows()]
    schwer = vollstaendig[schwer_maske]
    n_je_modell = {m: int((schwer[sp_] != "").sum())
                   for m, sp_ in modell_spalten.items()}
    print(f"Schwierige Einheiten (Menschen uneins): {len(schwer)} von "
          f"{len(vollstaendig)}")
    print("Auswertbare Basis je Modell: "
          + ", ".join(f"{m}={n}" for m, n in n_je_modell.items()))
    print()

    kopf = "Person".ljust(8) + "".join(m[:11].rjust(12) for m in modell_dateien)
    print("Differenz (Modell- minus Menschennaehe) auf schwierigen Einheiten:")
    print(kopf)
    for p in personen:
        andere = [q for q in personen if q != p]
        zeile = p.ljust(8)
        for mname, mspalte in modell_spalten.items():
            hat = schwer[schwer[mspalte] != ""]
            if len(hat) == 0:
                zeile += "n/a".rjust(12); continue
            pa_modell = (hat[spalte[p]] == hat[mspalte]).mean()
            pa_mensch = sum((hat[spalte[p]] == hat[spalte[q]]).mean()
                            for q in andere) / len(andere)
            diff = pa_modell - pa_mensch
            marke = ("*" if (diff > MODELL_SCHWELLE and len(hat) >= 10)
                     else " ")
            zeile += f"{diff:>+10.3f}{marke} "
            modellnaehe.setdefault(p, {})[mname] = dict(
                mit_modell=float(pa_modell), mit_menschen=float(pa_mensch),
                differenz=float(diff), n_schwer=int(len(hat)))
        print(zeile)
    print()
    print(f"* markiert Differenz ueber {MODELL_SCHWELLE:.2f} bei "
          "mindestens 10 auswertbaren Einheiten; kleinere Basen sind "
          "nicht belastbar.")

    auffaellig = [(p, m) for p in modellnaehe for m, v in modellnaehe[p].items()
                  if v["differenz"] > MODELL_SCHWELLE and v["n_schwer"] >= 10]
    print()
    if auffaellig:
        print("Auffaellige Naehe:")
        for p, m in auffaellig:
            print(f"  {p} steht dem Modell '{m}' naeher als den Menschen "
                  f"(Differenz {modellnaehe[p][m]['differenz']:+.3f}).")
        print("Angemessene Reaktion: im Gespraech einzelne Entscheidungen")
        print("nachfragen, insbesondere bei den schwierigen Einheiten.")
        print("Die Zahl allein ist kein Nachweis.")
    else:
        print("Keine Person zeigt auffaellige Naehe zu einem der Modelle.")

Schwierige Einheiten (Menschen uneins): 9 von 49
Auswertbare Basis je Modell: chatgpt-gpt5.6-luna_final=9, claude-fable5_final=9, claude-opus5_final=9

Differenz (Modell- minus Menschennaehe) auf schwierigen Einheiten:
Person   chatgpt-gpt claude-fabl claude-opus
A4          -0.111      +0.111      +0.111  
A2          -0.167      +0.167      +0.056  
A5          +0.389      +0.389      +0.278  

* markiert Differenz ueber 0.25 bei mindestens 10 auswertbaren Einheiten; kleinere Basen sind nicht belastbar.

Keine Person zeigt auffaellige Naehe zu einem der Modelle.


## Schritt 6 — Aufschlüsselung und Unsicherheit


In [7]:
import math
import numpy as np

vollstaendig["einheit_pa"] = [einheit_agreement(tuple(r))
                              for r in vollstaendig[sn].to_numpy()]
vollstaendig["einstimmig"] = vollstaendig["einheit_pa"] == 1.0

def quote(teil):
    return teil["einheit_pa"].mean() if len(teil) else float("nan")

for feld, titel in (("typ", "Pronomentyp"), ("sammlung", "Sammlung")):
    print(f"Nach {titel}:")
    for wert, teil in vollstaendig.groupby(feld):
        print(f"  {wert:<14} {quote(teil):.3f}  (n={len(teil)}, "
              f"nicht einstimmig={int((~teil['einstimmig']).sum())})")
    print()

klein_sonder = {w.casefold() for w in SONDERWERTE}
vollstaendig["kategorie"] = vollstaendig[spalte[HAUPTERHEBUNG]].map(
    lambda v: v if v in klein_sonder else "figurenname")
print("Nach Kategorie (Urteil der Haupterhebung):")
for k, teil in vollstaendig.groupby("kategorie"):
    print(f"  {k:<14} {quote(teil):.3f}  (n={len(teil)})")

rng = np.random.default_rng(SEED)
gruppen = [g["einheit_pa"].to_numpy()
           for _, g in vollstaendig.groupby("erzaehlung")]
idx = rng.integers(0, len(gruppen), size=(BOOTSTRAP_REPS, len(gruppen)))
werte = np.array([np.concatenate([gruppen[i] for i in reihe]).mean()
                  for reihe in idx])
lo, hi = np.percentile(werte, [2.5, 97.5])

p_quote = vollstaendig["einheit_pa"].mean()
n_einheiten = len(vollstaendig)
z = 1.96
nenner = 1 + z * z / n_einheiten
zentrum = (p_quote + z * z / (2 * n_einheiten)) / nenner
rand = z * math.sqrt(p_quote * (1 - p_quote) / n_einheiten
                     + z * z / (4 * n_einheiten * n_einheiten)) / nenner

print()
print(f"Uebereinstimmungsquote: {p_quote:.3f}  (n = {n_einheiten} Einheiten)")
print(f"  Cluster-Bootstrap ueber {len(gruppen)} Erzaehlungen: "
      f"[{lo:.3f}, {hi:.3f}]  (grob, wenige Cluster)")
print(f"  Wilson auf Einheitenebene: "
      f"[{zentrum - rand:.3f}, {zentrum + rand:.3f}]  (ignoriert Klumpung)")

Nach Pronomentyp:
  demonstrativ   0.667  (n=7, nicht einstimmig=3)
  personal       0.905  (n=42, nicht einstimmig=6)

Nach Sammlung:
  Andersen       0.879  (n=11, nicht einstimmig=2)
  Arabisch       0.806  (n=12, nicht einstimmig=3)
  Grimm          0.905  (n=14, nicht einstimmig=2)
  Japanisch      0.889  (n=12, nicht einstimmig=2)

Nach Kategorie (Urteil der Haupterhebung):
  expletiv       0.333  (n=4)
  figurenname    0.917  (n=36)
  kein_pronomen  0.333  (n=1)
  keine_figur    1.000  (n=8)

Uebereinstimmungsquote: 0.871  (n = 49 Einheiten)
  Cluster-Bootstrap ueber 4 Erzaehlungen: [0.826, 0.899]  (grob, wenige Cluster)
  Wilson auf Einheitenebene: [0.749, 0.938]  (ignoriert Klumpung)


## Schritt 7 — Divergenzen mit Mehrheitsvotum


In [8]:
unstimmig = vollstaendig[~vollstaendig["einstimmig"]].copy()

def mehrheit(zeile):
    stimmen = Counter(zeile[s] for s in sn)
    top, anzahl = stimmen.most_common(1)[0]
    if len(personen) > 2 and anzahl > len(personen) / 2:
        return top
    return ""

if len(unstimmig):
    unstimmig["mehrheit"] = unstimmig.apply(mehrheit, axis=1)
else:
    unstimmig["mehrheit"] = []

print(f"Divergenzen: {len(unstimmig)} von {len(vollstaendig)}")
if len(personen) > 2 and len(unstimmig):
    mit = int((unstimmig["mehrheit"] != "").sum())
    print(f"  davon mit Mehrheit entscheidbar: {mit}")
    print(f"  echte Grenzfaelle (alle uneins): {len(unstimmig) - mit}")
print()

for _, r in unstimmig.iterrows():
    print(f"[{r['sammlung']}] {r['erzaehlung']} | Satz {r['satzindex']}, "
          f"Token {r['token_nr']} | '{r['pronomen']}' ({r['typ']})")
    print("    " + " | ".join(f"{p}: {r[p]!r}" for p in personen))
    if r["mehrheit"]:
        print(f"    Mehrheit: {r['mehrheit']!r}")
    kontext = r["satz_markiert"]
    print(f"    {kontext[:110]}{'...' if len(kontext) > 110 else ''}")
    print()

spalten_export = (["sammlung", "erzaehlung", "tranche", "block", "satzindex",
                   "token_nr", "pronomen", "typ", "satz_markiert"]
                  + personen + ["mehrheit"])
export = unstimmig[spalten_export].copy()
export["entscheidung"] = ""
export["begruendung"] = ""
ziel = AUSGABE / f"divergenzen_pronomen_lauf{LAUF_NR:02d}.csv"
export.to_csv(ziel, index=False, encoding="utf-8")
print(f"Geschrieben: {ziel}")

print()
print("Haeufigste abweichende Urteilspaare (Haupterhebung gegen die anderen):")
paare_zaehler = Counter()
for _, r in unstimmig.iterrows():
    for p in zweitpersonen:
        if r[spalte[HAUPTERHEBUNG]] != r[spalte[p]]:
            paare_zaehler[(r[HAUPTERHEBUNG], r[p])] += 1
for (x, y), k in paare_zaehler.most_common(10):
    print(f"  {x!r:<26} -> {y!r:<26} {k}x")

Divergenzen: 9 von 49
  davon mit Mehrheit entscheidbar: 8
  echte Grenzfaelle (alle uneins): 1

[Andersen] 2_Des Kaisers neue Kleider | Satz 29, Token 21 | 'das' (demonstrativ)
    A4: 'kein_pronomen' | A2: 'keine_figur' | A5: 'keine_figur'
    Mehrheit: 'keine_figur'
    Der alte Minister merkte gut auf , damit er dasselbe sagen könne , wenn er zum Kaiser zurückkomme , und **das*...

[Andersen] 2_Des Kaisers neue Kleider | Satz 32, Token 17 | 'es' (personal)
    A4: 'expletiv' | A2: 'expletiv' | A5: 'keine_figur'
    Mehrheit: 'expletiv'
    Der Kaiser sandte bald wieder einen anderen tüchtigen Staatsmann hin , um zu sehen , wie **es** mit dem Weben ...

[Arabisch] 0_Die Geschichte von Ali dem Kairenser und dem spukenden Hause in Bagdad | Satz 110, Token 12 | 'ihnen' (personal)
    A4: 'Wachen' | A2: 'torwächter' | A5: 'Wachen'
    Mehrheit: 'wachen'
    er aß also mit den Wachen und schlief die Nacht bei **ihnen** .

[Arabisch] 0_Die Geschichte von Ali dem Kairenser und dem spukende

## Schritt 8 — Kommentare der Bearbeitenden einsammeln


In [9]:
basis_index = set(vollstaendig.index)
strittige_index = set(unstimmig.index)
erz_je_index = dict(zip(vollstaendig.index,
                        vollstaendig["erzaehlung"]))
pron_je_index = dict(zip(vollstaendig.index,
                         vollstaendig["pronomen"]))

kommentare = []
for person in personen:
    spalte = tabellen[person].get("kommentar")
    if spalte is None:
        continue
    for idx, text in spalte.items():
        text = str(text).strip()
        if not text or idx not in basis_index:
            continue
        kommentare.append({
            "person": person,
            "erzaehlung": erz_je_index[idx],
            "pronomen": pron_je_index[idx],
            "strittig": bool(idx in strittige_index),
            "kommentar": text,
        })

print(f"Kommentare insgesamt: {len(kommentare)}")
bei_strittig = [k for k in kommentare if k["strittig"]]
bei_einig    = [k for k in kommentare if not k["strittig"]]
print(f"  bei strittigen Einheiten:   {len(bei_strittig)}")
print(f"  bei einstimmigen Einheiten: {len(bei_einig)}  <- besonders pruefen")

if bei_einig:
    print()
    print("Kommentare trotz Einstimmigkeit (moegliche Regelluecke):")
    for k in bei_einig[:15]:
        print(f"  [{k['person']}] {k['pronomen']}: {k['kommentar'][:70]}")
if bei_strittig:
    print()
    print("Kommentare bei strittigen Einheiten (Auszug):")
    for k in bei_strittig[:10]:
        print(f"  [{k['person']}] {k['pronomen']}: {k['kommentar'][:70]}")

if kommentare:
    ziel_k = AUSGABE / f"kommentare_pronomen_lauf{LAUF_NR:02d}.csv"
    pd.DataFrame(kommentare).to_csv(ziel_k, index=False, encoding="utf-8")
    print(f"\nGeschrieben: {ziel_k}")
else:
    print("\nKeine Kommentare hinterlegt.")


Kommentare insgesamt: 61
  bei strittigen Einheiten:   10
  bei einstimmigen Einheiten: 51  <- besonders pruefen

Kommentare trotz Einstimmigkeit (moegliche Regelluecke):
  [A4] dasselbe: Bezieht sich auf das Gesagte
  [A4] das: Referiert auf Dinar
  [A4] es: Referiert auf Dinar
  [A4] ihn: Referiert auf Dinar
  [A4] ihn: Referiert auf Dinar
  [A2] er: damit "er" dasselbe  →  Minister
  [A2] dasselbe: Demonstrativpronomen. Bezieht sich auf die Aussage der Weber
  [A2] er: Wenn *er* zum → Minister
  [A2] er: das tat "er" → Minister
  [A2] es: um "es" zum → Bezieht sich auf die angeforderte Ware
  [A2] Sie: "sie" steckten alles → referiert auf die Betrüger, die sich als Weber 
  [A2] sie: Aber "sie" fuhren → referiert auf die Betrüger, die sich als Weber aus
  [A2] er: aber "er" hat → der Kaiser
  [A2] er: aber "er" hat → der Kaiser
  [A2] das: Nimm "das" → bezieht sich auf den Dinar, den Ali einem Torwächter gibt

Kommentare bei strittigen Einheiten (Auszug):
  [A2] das: Demonstrativpro

## Schritt 9 — Ergebnis sichern und einordnen


In [10]:
import json
from datetime import datetime

ergebnis = {
    "lauf": LAUF_NR,
    "etikett": LAUF_ETIKETT,
    "zeitpunkt": datetime.now().isoformat(timespec="minutes"),
    "ebene": "pronomen",
    "bearbeitende": personen,
    "messgrundlage": {"n_verglichen": int(len(vollstaendig)),
                      "n_mind_eine_zweitperson": int(len(bearbeitet)),
                      "erzaehlungen": erz},
    "streng": streng,
    "normalisiert": normal,
    "hauptauswertung": "normalisiert" if GROSSKLEIN_EGAL else "streng",
    "quote": float(p_quote),
    "cluster_bootstrap_95": [float(lo), float(hi)],
    "wilson_95": [float(zentrum - rand), float(zentrum + rand)],
    "paarweise": paarwerte,
    "abweichungsindex": {k: float(v) for k, v in index.items()},
    "modellnaehe": modellnaehe,  # {person: {modell: {differenz, ...}}}
    "modelle": list(modell_dateien.keys()),
    "nach_typ": {t: float(quote(g)) for t, g in vollstaendig.groupby("typ")},
    "nach_sammlung": {s: float(quote(g))
                      for s, g in vollstaendig.groupby("sammlung")},
    "nach_kategorie": {k: float(quote(g))
                       for k, g in vollstaendig.groupby("kategorie")},
    "divergenzen": int(len(unstimmig)),
    "nur_schreibweise": int(nur_schreibweise),
}
ziel_json = AUSGABE / f"iaa_pronomen_lauf{LAUF_NR:02d}.json"
ziel_json.write_text(json.dumps(ergebnis, ensure_ascii=False, indent=2),
                     encoding="utf-8")
print(f"Geschrieben: {ziel_json}")

a_wert = haupt["fleiss_kappa"]
print()
print(f"Fleiss' kappa = {a_wert:.3f}  ({len(personen)} Bearbeitende, "
      f"beobachtet {haupt['beobachtet']:.3f}, "
      f"erwartet {haupt['erwartet']:.3f})")
if a_wert >= 0.8:
    print("  Einordnung: gute Reliabilitaet (Konvention: ab 0,8).")
elif a_wert >= 0.67:
    print("  Einordnung: nur vorsichtige Schluesse zulaessig (0,67 bis 0,8).")
else:
    print("  Einordnung: unzureichend — Richtlinie und Einweisung pruefen,")
    print("  bevor die Haupterhebung weiterverwendet wird.")

print()
print("Alle bisherigen Laeufe:")
for pfad in sorted(AUSGABE.glob("iaa_pronomen_lauf*.json")):
    d = json.loads(pfad.read_text(encoding="utf-8"))
    print(f"  Lauf {d['lauf']:>2} | {d['zeitpunkt']} | {d['etikett']:<22} | "
          f"{len(d['bearbeitende'])} Pers. | Quote {d['quote']:.3f} | "
          f"Fleiss K {d['normalisiert']['fleiss_kappa']:.3f} | "
          f"Divergenzen {d['divergenzen']}")
if LAUF_NR > 1:
    print()
    print("HINWEIS: Als Reliabilitaetswert massgeblich ist der Lauf VOR der")
    print("Adjudikation (Lauf 1). Spaetere Laeufe dokumentieren die")
    print("Konvergenz nach Korrekturen, ersetzen die Messung aber nicht.")

Geschrieben: ../05_ergebnisse/iaa/iaa_pronomen_lauf01.json

Fleiss' kappa = 0.858  (3 Bearbeitende, beobachtet 0.871, erwartet 0.090)
  Einordnung: gute Reliabilitaet (Konvention: ab 0,8).

Alle bisherigen Laeufe:
  Lauf  1 | 2026-09-13T19:04 | vor Adjudikation       | 3 Pers. | Quote 0.871 | Fleiss K 0.858 | Divergenzen 9
